# Latihan Mandiri - Primitif Sinkronisasi

**Nama:** Afelia Nasyifa  
**NIM:** 2411513010  
**Mata kuliah:** Pemrograman Paralel

## 1. RLock pada Class RekeningBank

Buat kelas `RekeningBank` yang memiliki method `setor(jumlah)`, `tarik(jumlah)`, dan `transfer(rekening_tujuan, jumlah)`. Saat transfer, rekening asal memanggil `tarik()` dan rekening tujuan memanggil `setor()`. Saya menggunakan `threading.RLock` untuk tiap rekening karena satu thread perlu mengambil lock yang sama lagi ketika method di dalam class saling memanggil.

In [1]:
import threading

class RekeningBank:
    def __init__(self, nama, saldo_awal=0):
        self.nama = nama
        self.saldo = saldo_awal
        self.lock = threading.RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setoran harus lebih dari 0")
        with self.lock:
            self.saldo += jumlah
            return self.saldo

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah penarikan harus lebih dari 0")
        with self.lock:
            if self.saldo < jumlah:
                raise ValueError(f"Saldo rekening {self.nama} tidak mencukupi")
            self.saldo -= jumlah
            return self.saldo

    def transfer(self, rekening_tujuan, jumlah):
        if not isinstance(rekening_tujuan, RekeningBank):
            raise TypeError("Tujuan harus berupa objek RekeningBank")
        if rekening_tujuan is self:
            raise ValueError("Tidak bisa transfer ke rekening sendiri")
        if jumlah <= 0:
            raise ValueError("Jumlah transfer harus lebih dari 0")
        # Kedua lock selalu diambil berdasarkan urutan id objek.
        # Ini mencegah deadlock ketika dua rekening transfer dua arah.
        rekening_pertama, rekening_kedua = sorted(
            [self, rekening_tujuan], key=id
        )
        with rekening_pertama.lock:
            with rekening_kedua.lock:
                self.tarik(jumlah)                 # mengambil RLock asal lagi
                rekening_tujuan.setor(jumlah)     # mengambil RLock tujuan lagi
                print(f"{self.nama} -> {rekening_tujuan.nama}: Rp{jumlah:,}")

    def lihat_saldo(self):
        with self.lock:
            return self.saldo

rekening_a = RekeningBank("A", 500_000)
rekening_b = RekeningBank("B", 200_000)
print("Saldo awal:", rekening_a.lihat_saldo(), rekening_b.lihat_saldo())
rekening_a.transfer(rekening_b, 100_000)
rekening_b.transfer(rekening_a, 50_000)
print("Saldo akhir:", rekening_a.lihat_saldo(), rekening_b.lihat_saldo())
assert (rekening_a.lihat_saldo(), rekening_b.lihat_saldo()) == (450_000, 250_000)
assert rekening_a.lihat_saldo() + rekening_b.lihat_saldo() == 700_000


Saldo awal: 500000 200000
A -> B: Rp100,000
B -> A: Rp50,000
Saldo akhir: 450000 250000


**Penjelasan:** `RLock` diperlukan karena `transfer()` memegang lock rekening, kemudian memanggil `tarik()` atau `setor()` yang mengambil lock rekening tersebut lagi. `Lock` biasa tidak dapat diambil ulang oleh thread yang sama, sehingga program bisa macet (*self-deadlock*). `RLock` memperbolehkan pengambilan berulang oleh thread pemiliknya. Untuk transfer dua arah, lock kedua rekening juga diambil dengan urutan yang sama agar tidak terjadi deadlock antar-rekening. Jika saldo kurang, penarikan ditolak sebelum setoran dilakukan.

## 2. Deadlock dengan Tiga Lock

Pola deadlock yang diminta:

- Thread1 memegang Lock-A, lalu menunggu Lock-B.
- Thread2 memegang Lock-B, lalu menunggu Lock-C.
- Thread3 memegang Lock-C, lalu menunggu Lock-A.

Ketiganya menunggu secara melingkar (*circular wait*). Untuk membuktikannya tanpa membuat notebook macet, program deadlock dijalankan di proses terpisah, lalu dihentikan jika melewati batas waktu.

In [2]:
import subprocess
import sys
import textwrap

kode_deadlock = textwrap.dedent(r"""
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()
barrier = threading.Barrier(3)

def tugas(nama, lock_pertama, lock_kedua, nama_kedua):
    with lock_pertama:
        print(f"{nama} memegang lock pertama", flush=True)
        barrier.wait()  # pastikan ketiga thread sudah memegang lock pertama
        print(f"{nama} menunggu {nama_kedua}", flush=True)
        with lock_kedua:
            print(f"{nama} selesai", flush=True)

threads = [
    threading.Thread(target=tugas, args=("Thread1", lock_a, lock_b, "Lock-B")),
    threading.Thread(target=tugas, args=("Thread2", lock_b, lock_c, "Lock-C")),
    threading.Thread(target=tugas, args=("Thread3", lock_c, lock_a, "Lock-A")),
]
for t in threads:
    t.start()
for t in threads:
    t.join()
""")

try:
    proses = subprocess.run(
        [sys.executable, "-u", "-c", kode_deadlock],
        capture_output=True, text=True, timeout=2
    )
    print(proses.stdout)
    print("Proses selesai (tidak terjadi deadlock)")
except subprocess.TimeoutExpired as error:
    keluaran = error.stdout or b""
    if isinstance(keluaran, bytes):
        keluaran = keluaran.decode("utf-8", errors="replace")
    print(keluaran)
    print("DEADLOCK TERBUKTI: ketiga thread tidak selesai dalam 2 detik.")
    print("Subprocess dihentikan otomatis, kernel notebook tetap berjalan.")


Thread1 memegang lock pertamaThread2 memegang lock pertama

Thread3 memegang lock pertama
Thread3 menunggu Lock-A
Thread2 menunggu Lock-C
Thread1 menunggu Lock-B

DEADLOCK TERBUKTI: ketiga thread tidak selesai dalam 2 detik.
Subprocess dihentikan otomatis, kernel notebook tetap berjalan.


**Perbaikan:** Semua thread harus mengambil lock dengan urutan yang konsisten, yaitu A → B → C (sesuai lock yang diperlukan). Dengan begitu, tidak ada thread yang memegang lock urutan belakang sambil menunggu lock urutan depan, sehingga *circular wait* terputus.

In [3]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

# Semua thread mengambil lock sesuai urutan global A, B, C.
urutan_lock = {"A": (0, lock_a), "B": (1, lock_b), "C": (2, lock_c)}

def tugas_aman(nama, lock_dibutuhkan):
    lock_urut = sorted((urutan_lock[x] for x in lock_dibutuhkan), key=lambda x: x[0])
    for _, lock in lock_urut:
        lock.acquire()
    try:
        print(f"{nama} berhasil memegang {', '.join(sorted(lock_dibutuhkan))}")
        time.sleep(0.05)
    finally:
        for _, lock in reversed(lock_urut):
            lock.release()
    print(f"{nama} selesai")

threads = [
    threading.Thread(target=tugas_aman, args=("Thread1", ["A", "B"])),
    threading.Thread(target=tugas_aman, args=("Thread2", ["B", "C"])),
    threading.Thread(target=tugas_aman, args=("Thread3", ["C", "A"])),
]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=3)
assert all(not t.is_alive() for t in threads), "Ada thread yang belum selesai"
print("Tidak ada deadlock: semua thread selesai.")


Thread1 berhasil memegang A, B
Thread1 selesai
Thread2 berhasil memegang B, C
Thread2 selesai
Thread3 berhasil memegang A, C
Thread3 selesai
Tidak ada deadlock: semua thread selesai.


**Kesimpulan:** Sebelum diperbaiki, proses berhenti karena setiap thread menunggu lock yang sedang dipegang thread lain. Setelah urutan pengambilan lock dibuat konsisten, ketiga thread selesai. `timeout` pada `subprocess` digunakan untuk menguji deadlock, bukan untuk memperbaiki penyebabnya.

## 3. Bounded Buffer dengan Kapasitas Berbeda

Saya mengulang pola dari materi: dua semaphore (`slot_kosong` dan `slot_terisi`) serta satu mutex, kali ini memakai 2 producer + 2 consumer. Jumlah item total 20 (masing-masing producer membuat 10 item, masing-masing consumer mengambil 10 item). Percobaan dilakukan untuk `UKURAN_BUFFER` = 1, 3, dan 10. Kecepatan producer dibuat lebih tinggi supaya pengaruh ukuran buffer terlihat.

In [4]:
import threading
import time


def jalankan_buffer(ukuran_buffer, item_per_producer=10):
    buffer = []
    mutex = threading.Lock()
    slot_kosong = threading.Semaphore(ukuran_buffer)
    slot_terisi = threading.Semaphore(0)

    hasil_produksi = []
    hasil_konsumsi = []
    waktu_tunggu_producer = []
    waktu_tunggu_consumer = []
    kapasitas_tertinggi = [0]
    catatan = []

    def producer(nama):
        for i in range(item_per_producer):
            item = f"{nama}-item-{i}"
            time.sleep(0.002)  # produksi relatif cepat
            awal = time.perf_counter()
            slot_kosong.acquire()  # menunggu jika buffer penuh
            tunggu = time.perf_counter() - awal
            with mutex:
                buffer.append(item)
                hasil_produksi.append(item)
                kapasitas_tertinggi[0] = max(kapasitas_tertinggi[0], len(buffer))
                if len(catatan) < 8:
                    catatan.append(f"[{nama}] taruh {item} (buffer: {len(buffer)}/{ukuran_buffer})")
                waktu_tunggu_producer.append(tunggu)
            slot_terisi.release()

    def consumer(nama):
        for _ in range(item_per_producer):
            awal = time.perf_counter()
            slot_terisi.acquire()  # menunggu jika buffer kosong
            tunggu = time.perf_counter() - awal
            with mutex:
                item = buffer.pop(0)
                hasil_konsumsi.append(item)
                if len(catatan) < 8:
                    catatan.append(f"[{nama}] ambil {item} (buffer: {len(buffer)}/{ukuran_buffer})")
                waktu_tunggu_consumer.append(tunggu)
            slot_kosong.release()
            time.sleep(0.018)  # konsumsi lebih lambat daripada produksi

    threads = [
        threading.Thread(target=producer, args=("Producer-1",)),
        threading.Thread(target=producer, args=("Producer-2",)),
        threading.Thread(target=consumer, args=("Consumer-1",)),
        threading.Thread(target=consumer, args=("Consumer-2",)),
    ]
    awal_percobaan = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=5)
    durasi = time.perf_counter() - awal_percobaan

    assert all(not t.is_alive() for t in threads), "Ada thread belum selesai"
    assert len(buffer) == 0, "Buffer seharusnya kosong"
    assert len(hasil_produksi) == len(hasil_konsumsi) == 2 * item_per_producer
    assert sorted(hasil_produksi) == sorted(hasil_konsumsi)
    assert kapasitas_tertinggi[0] <= ukuran_buffer

    return {
        "kapasitas": ukuran_buffer,
        "maks_terisi": kapasitas_tertinggi[0],
        "total_item": len(hasil_konsumsi),
        "durasi": durasi,
        "tunggu_producer": sum(waktu_tunggu_producer),
        "tunggu_consumer": sum(waktu_tunggu_consumer),
        "producer_tertahan": sum(w > 0.003 for w in waktu_tunggu_producer),
        "log": catatan,
    }

hasil_percobaan = [jalankan_buffer(k) for k in (1, 3, 10)]
for hasil in hasil_percobaan:
    print(f"\n=== UKURAN_BUFFER = {hasil['kapasitas']} ===")
    for baris in hasil["log"]:
        print(baris)
    print(f"Total dikonsumsi: {hasil['total_item']} | Maks. isi: {hasil['maks_terisi']}")
    print(f"Total waktu tunggu producer: {hasil['tunggu_producer']:.4f} detik")
    print(f"Producer tertahan > 3 ms: {hasil['producer_tertahan']} kali")
    print(f"Sisa buffer: 0 | Durasi: {hasil['durasi']:.4f} detik")



=== UKURAN_BUFFER = 1 ===
[Producer-1] taruh Producer-1-item-0 (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-0 (buffer: 0/1)
[Producer-2] taruh Producer-2-item-0 (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-0 (buffer: 0/1)
[Producer-1] taruh Producer-1-item-1 (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-1 (buffer: 0/1)
[Producer-2] taruh Producer-2-item-1 (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-1 (buffer: 0/1)
Total dikonsumsi: 20 | Maks. isi: 1
Total waktu tunggu producer: 0.2712 detik
Producer tertahan > 3 ms: 17 kali
Sisa buffer: 0 | Durasi: 0.1849 detik

=== UKURAN_BUFFER = 3 ===
[Producer-1] taruh Producer-1-item-0 (buffer: 1/3)
[Consumer-1] ambil Producer-1-item-0 (buffer: 0/3)
[Producer-2] taruh Producer-2-item-0 (buffer: 1/3)
[Consumer-2] ambil Producer-2-item-0 (buffer: 0/3)
[Producer-1] taruh Producer-1-item-1 (buffer: 1/3)
[Producer-2] taruh Producer-2-item-1 (buffer: 2/3)
[Producer-1] taruh Producer-1-item-2 (buffer: 3/3)
[Consumer-1] ambil Producer-1-i

### Ringkasan perbandingan

Waktu tunggu dihitung dari lamanya producer menunggu `slot_kosong.acquire()` dan consumer menunggu `slot_terisi.acquire()`. Nilainya bisa berubah sedikit pada setiap eksekusi karena penjadwalan thread.

In [5]:
print(f"{'Buffer':>8} | {'Maks isi':>8} | {'Tunggu producer (s)':>20} | {'Tertahan >3ms':>13} | {'Item selesai':>12}")
print("-" * 78)
for h in hasil_percobaan:
    print(f"{h['kapasitas']:>8} | {h['maks_terisi']:>8} | {h['tunggu_producer']:>20.4f} | {h['producer_tertahan']:>13} | {h['total_item']:>12}")


  Buffer | Maks isi |  Tunggu producer (s) | Tertahan >3ms | Item selesai
------------------------------------------------------------------------------
       1 |        1 |               0.2712 |            17 |           20
       3 |        3 |               0.2344 |            15 |           20
      10 |       10 |               0.1079 |             8 |           20


**Pengamatan:**

- **Buffer 1:** producer lebih sering berhenti karena hanya ada satu slot. Begitu buffer terisi, producer harus menunggu consumer mengambil item.
- **Buffer 3:** producer masih menunggu ketika tiga slot terisi, tetapi ada lebih banyak ruang untuk menampung item sementara.
- **Buffer 10:** producer dapat menaruh lebih banyak item sebelum menunggu. Umumnya total waktu tunggu producer lebih kecil, meskipun waktu penyelesaian seluruh item tetap bergantung pada kecepatan consumer.

Pada ketiga ukuran, semua 20 item berhasil dikonsumsi, buffer akhir kosong, dan jumlah item di buffer tidak melampaui kapasitas. Hasil angka pastinya terlihat pada tabel output.

## 4. Refleksi Singkat

Menurut saya, deadlock paling berbahaya untuk aplikasi produksi seperti sistem perbankan atau e-commerce. Saat deadlock terjadi, beberapa proses bisa berhenti total karena saling menunggu resource. Contohnya, transaksi transfer atau pembaruan stok bisa tertahan dan membuat pengguna tidak dapat menyelesaikan transaksi. Masalah ini perlu cepat dideteksi karena dari luar aplikasi bisa terlihat seperti hanya sedang lambat, padahal prosesnya tidak akan selesai sendiri. Cara pencegahannya adalah menetapkan urutan pengambilan lock yang konsisten, membatasi waktu tunggu, dan memantau proses yang terlalu lama. Livelock dan starvation juga mengganggu, tetapi deadlock lebih langsung menghentikan pekerjaan yang sedang berlangsung.